In [4]:
## import some of needed libraries

import pandas as pd
import numpy as np

In [5]:
## read data
data = pd.read_csv('../en_fr_train.csv')
data.head()

,en,fr
0,"Thank you so much, Chris. And it's truly a gre...","Merci beaucoup, Chris. C'est vraiment un honne..."
1,"I have been blown away by this conference, and...",J'ai été très impressionné par cette conférenc...
2,"And I say that sincerely, partly because Put ...","Et je dis çà sincèrement, en autres parce que ..."
3,I flew on Air Force Two for eight years.,J'ai volé avec Air Force 2 pendant huit ans.
4,Now I have to take off my shoes or boots to ge...,"Et maintenant, je dois enlever mes chaussures ..."


In [6]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 232825 entries, 0 to 232824
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   en      232825 non-null  str  
 1   fr      232825 non-null  str  
dtypes: str(2)
memory usage: 48.8 MB


In [7]:
data['en'] = data['en'].str.lower()
data['fr'] = data['fr'].str.lower()

In [8]:
data

,en,fr
0,"thank you so much, chris. and it's truly a gre...","merci beaucoup, chris. c'est vraiment un honne..."
1,"i have been blown away by this conference, and...",j'ai été très impressionné par cette conférenc...
2,"and i say that sincerely, partly because put ...","et je dis çà sincèrement, en autres parce que ..."
3,i flew on air force two for eight years.,j'ai volé avec air force 2 pendant huit ans.
4,now i have to take off my shoes or boots to ge...,"et maintenant, je dois enlever mes chaussures ..."
...,...,...
232820,"it was about the experience, about the story, ...","c'était pour l'expérience, pour l'histoire, po..."
232821,"from the streets of the neighborhood, the pain...","des rues du quartier, la peinture apparaît en ..."
232822,but connected with the sign of calligraphy tha...,"mais connectés avec la calligraphie, ils révèl..."
232823,anyone who wants to see the sunlight clearly n...,quiconque voulant voir clairement la lumière d...


In [9]:
## adding SOS and EOS
data["en"] = data["en"].apply(
    lambda x: "<sos> " + x + " <eos>"
)
data["fr"] = data["fr"].apply(
    lambda x: "<sos> " + x + " <eos>"
)

In [10]:
## now we are doing tokenize
from tensorflow.keras.preprocessing.text import Tokenizer

en_tokenizer = Tokenizer(num_words=10000)
fr_tokenizer = Tokenizer(num_words=10000)

en_tokenizer.fit_on_texts(data["en"])
fr_tokenizer.fit_on_texts(data["fr"])

In [11]:
en_voc_size = len(en_tokenizer.word_index)+1
fr_voc_size = len(fr_tokenizer.word_index)+1

In [12]:
en_sequences = en_tokenizer.texts_to_sequences(data["en"])
fr_sequences = fr_tokenizer.texts_to_sequences(data["fr"])

In [13]:
# en_sequences

In [14]:
max_sequence_en_len = max([len(i) for i in en_sequences])
max_sequence_fr_len = max([len(i) for i in fr_sequences])

In [15]:
# max_sequence_en_len  # 95
max_sequence_fr_len

92

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

en_sequences = pad_sequences(en_sequences,maxlen=max_sequence_en_len,padding='pre')
fr_sequences = pad_sequences(fr_sequences,maxlen=max_sequence_fr_len,padding='pre')

In [17]:
en_sequences

array([[   0,    0,    0, ...,  939, 3092,    1],
       [   0,    0,    0, ...,   89,  480,    1],
       [   0,    0,    0, ...,   35, 1185,    1],
       ...,
       [   0,    0,    0, ..., 2598,  481,    1],
       [   0,    0,    0, ...,  575,  102,    1],
       [   0,    0,    0, ...,  160,   12,    1]], dtype=int32)

In [18]:
X = en_sequences
y = fr_sequences

In [19]:
## Train Test and Split
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [20]:
decoder_input = y_train[:, :-1]
decoder_target = y_train[:, 1:]

In [21]:
decoder_input_test = y_test[:, :-1]
decoder_target_test = y_test[:, 1:]

In [22]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding , Dense, Dropout,LSTM

In [23]:
## we fix embedding and hidden dimention
embedding_dim = 50
hidden_dim = 75

In [24]:
## now going to frist Encoder

# take inputs
encoder_input = Input(shape=(None,))

# Embendding
encoder_embedding = Embedding(en_voc_size,embedding_dim)(encoder_input)

# LSTM model make
encoder_lstm = LSTM(hidden_dim,return_state=True)


encoder_output, hidden_state, cell_state = encoder_lstm(
    encoder_embedding
)


In [25]:
## now decoder

# take inputs of fr section
decoder_input = Input(shape=(None,))

#then embedding
decoder_embedding = Embedding(
    fr_voc_size,
    embedding_dim
)(decoder_input)

# LSTM model prepare 
decoder_lstm = LSTM(
    hidden_dim,
    return_sequences=True,
    return_state=True
)

decoder_output, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[hidden_state, cell_state]
)

In [26]:
# activation function apply on o/p of decoder
decoder_dense = Dense(
    fr_voc_size,
    activation="softmax"
)

decoder_output = decoder_dense(decoder_output)

In [27]:
# create a model

model = Model(
    [encoder_input,decoder_input],
    decoder_output
)

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])


In [28]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 50)  │  2,798,050 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 50)  │  4,249,400 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 75),      │     37,800 │ embedding[0][0]   │
│                     │ (None, 75),       │            │                   │
│                     │ (None, 75)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     37,800 │ embedding_1[0][0… │
│                     │ 75), (None, 75),  │            │ lstm[0][1],       │
│                     │ (None, 75)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  6,459,088 │ lstm_1[0][0]      │
│                     │ 84988)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,582,138 (51.81 MB)

 Trainable params: 13,582,138 (51.81 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
print(len(en_tokenizer.word_index))
print(len(fr_tokenizer.word_index))

55960
84987


In [ ]:
# history = model.fit(
#     [X_train, decoder_input],
#     decoder_target,
#     batch_size=64,
#     epochs=20,
#     validation_data=(
#         [X_test, decoder_input_test],
#         decoder_target_test
#     )
# )

In [1]:
from tensorflow.keras.models import load_model

model = load_model('en_to_fr_translation.h5')

In [2]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 50)  │  2,798,050 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 50)  │  4,249,400 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 75),      │     37,800 │ embedding[0][0]   │
│                     │ (None, 75),       │            │                   │
│                     │ (None, 75)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │     37,800 │ embedding_1[0][0… │
│                     │ 75), (None, 75),  │            │ lstm[0][1],       │
│                     │ (None, 75)]       │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │  6,459,088 │ lstm_1[0][0]      │
│                     │ 84988)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,582,140 (51.81 MB)

 Trainable params: 13,582,138 (51.81 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)